# M8b · Handling Outliers

**Goal:** detect and treat outliers in the cleaned feature matrix produced
by `02_handle_missing_values.ipynb`. Outliers can dominate `mean()`-based
features and skew tree-split criteria; left untreated they often manifest
as suspiciously high feature importance for a single column.

**Three techniques covered:**

1. **Z-score** — flags points > 3σ. Good baseline for ~Gaussian features.
2. **IQR rule** — flags points outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.
   Robust to skew; the standard for boxplots.
3. **Winsorization** — *clips* extreme values to a percentile (e.g. 1st/99th)
   instead of dropping rows. Preferred when sample size is small.

**Exit criteria:**
- A printed table of outlier counts per technique per feature.
- A *winsorized* DataFrame saved to `data/ml/step2_winsorized.parquet`.
- Distribution plots showing pre/post for two representative features.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — load the post-imputation matrix

In [ ]:
import pandas as pd, pathlib
df = pd.read_parquet('../../data/ml/step1_no_missing.parquet')
print(f'rows={len(df):,}  cols={df.shape[1]}')

# Identifier columns that we EXCLUDE from outlier treatment.
ID_COLS = ['tenant_id', 'device_id', 'year_month']
NUM_COLS = [c for c in df.select_dtypes('number').columns if c not in ID_COLS]
print('numeric features:', len(NUM_COLS))


## 3. Detect — Z-score and IQR side by side

In [ ]:
import numpy as np

def zscore_outliers(s: pd.Series, k: float = 3.0) -> int:
    z = (s - s.mean()) / s.std(ddof=0)
    return int((z.abs() > k).sum())

def iqr_outliers(s: pd.Series, k: float = 1.5) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    lo, hi = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
    return int(((s < lo) | (s > hi)).sum())

report = pd.DataFrame({
    'zscore_3sd': [zscore_outliers(df[c]) for c in NUM_COLS],
    'iqr_1.5':    [iqr_outliers(df[c])    for c in NUM_COLS],
}, index=NUM_COLS).sort_values('iqr_1.5', ascending=False)
display(report.head(20))


## 4. Treat — Winsorize at 1st / 99th percentile

Clipping (rather than dropping) preserves N and is the safer default for
small datasets. Tree-based models care about *order*, not absolute values,
so winsorization rarely hurts XGBoost / LightGBM / RandomForest.


In [ ]:
df_w = df.copy()
for col in NUM_COLS:
    lo, hi = df_w[col].quantile([0.01, 0.99])
    df_w[col] = df_w[col].clip(lower=lo, upper=hi)

# Sanity: winsorized series cannot exceed clip bounds.
breaches = []
for col in NUM_COLS:
    lo, hi = df[col].quantile([0.01, 0.99])
    if df_w[col].max() > hi + 1e-9 or df_w[col].min() < lo - 1e-9:
        breaches.append(col)
assert not breaches, f'clip breaches: {breaches}'
print('OK — all features winsorized to [P1, P99].')


## 5. Visualize before/after for two representative features

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax_row, col in zip(axes, ['p95_max_speed', 'harsh_events_per_100km']):
    if col not in df.columns:
        continue
    ax_row[0].hist(df[col], bins=60, alpha=0.7);    ax_row[0].set_title(f'{col} — raw')
    ax_row[1].hist(df_w[col], bins=60, alpha=0.7);  ax_row[1].set_title(f'{col} — winsorized')
plt.tight_layout(); plt.show()


## 6. Persist

In [ ]:
out = pathlib.Path('../../data/ml/step2_winsorized.parquet')
df_w.to_parquet(out, index=False)
print('saved to', out.resolve())
